# # Film Eleştirileri ve Bag-of-Words Modellemesi

🎯 Bu zorluğun amacı, metinlerin ***Bag-of-words*** modellemesiyle oynamaktır.

✍️ Aşağıdaki veri setinde, _“olumlu”_ veya _“olumsuz”_ olarak sınıflandırılmış 2000 adet yorum bulunmaktadır.

In [1]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/movie_reviews.csv")
data.head()

,target,reviews
0,neg,"plot : two teen couples go to a church party ,..."
1,neg,the happy bastard's quick movie review \ndamn ...
2,neg,it is movies like these that make a jaded movi...
3,neg,""" quest for camelot "" is warner bros . ' firs..."
4,neg,synopsis : a mentally unstable man undergoing ...


In [2]:
data.shape

(2000, 2)

## 1. Ön işleme (Preprocessing)

❓ **Soru (Metin Temizleme)** ❓

- Bir cümleyi temizleyecek bir `preprocessing` fonksiyonu yazın ve bunu tüm yorumlara uygulayın. Fonksiyon şunları yapmalıdır:
    - boşlukları kaldırma
    - harfleri küçük harfe çevirme
    - sayıları kaldırma
    - noktalama işaretlerini kaldırma
    - tokenization (kelimelere ayırma)
    - lemmatization (kelime köküne indirgeme)
- Temizlenmiş yorumları `clean_reviews` adlı bir sütunda saklayabilirsiniz.
- Bu aşamada stopword’leri kaldırmayın; nedenini `3. N-gram modelleme` bölümünde açıklayacağız.

In [7]:
import string
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer

import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

def preprocessing(sentence):
    sentence = sentence.lower()

    sentence = re.sub(r'\d+', '', sentence)

    sentence = sentence.translate(str.maketrans('', '', string.punctuation))

    sentence = sentence.strip()

    tokens = word_tokenize(sentence)

    lemmatizer = WordNetLemmatizer()

    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(cleaned_tokens)

[nltk_data] Downloading package punkt to
[nltk_data]     /home/fukansimsek/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/fukansimsek/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/fukansimsek/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [8]:
import re

data['clean_reviews'] = data['reviews'].apply(preprocessing)

print(data[['reviews', 'clean_reviews']].head())

                                             reviews  \
0  plot : two teen couples go to a church party ,...   
1  the happy bastard's quick movie review \ndamn ...   
2  it is movies like these that make a jaded movi...   
3   " quest for camelot " is warner bros . ' firs...   
4  synopsis : a mentally unstable man undergoing ...   

                                       clean_reviews  
0  plot two teen couple go to a church party drin...  
1  the happy bastard quick movie review damn that...  
2  it is movie like these that make a jaded movie...  
3  quest for camelot is warner bros first feature...  
4  synopsis a mentally unstable man undergoing ps...  


❓ **Soru (LabelEncoding)**❓

Hedefinizi LabelEncode ile kodlayın ve `“target_encoded”` adlı bir sütuna kaydedin.

In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

data['target_encoded'] = le.fit_transform(data['target'])

print(data[['target', 'target_encoded']].head())
print("\nSınıf Eşleşmeleri:")
for index, label in enumerate(le.classes_):
    print(f"{label} -> {index}")

  target  target_encoded
0    neg               0
1    neg               0
2    neg               0
3    neg               0
4    neg               0

Sınıf Eşleşmeleri:
neg -> 0
pos -> 1


In [10]:
# Hızlı kontrol
data.head()

,target,reviews,clean_reviews,target_encoded
0,neg,"plot : two teen couples go to a church party ,...",plot two teen couple go to a church party drin...,0
1,neg,the happy bastard's quick movie review \ndamn ...,the happy bastard quick movie review damn that...,0
2,neg,it is movies like these that make a jaded movi...,it is movie like these that make a jaded movie...,0
3,neg,""" quest for camelot "" is warner bros . ' firs...",quest for camelot is warner bros first feature...,0
4,neg,synopsis : a mentally unstable man undergoing ...,synopsis a mentally unstable man undergoing ps...,0


## 2. Bag-of-Words Modellemesi

❓ **Soru (Tek kelimelik sözcüklerle NaiveBayes)** ❓

`cross_validate` kullanarak, metinlerin Bag-of-Words temsilinde eğitilmiş bir Multinomial Naive Bayes modelini puanlayın.

In [11]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_validate

vectorizer = CountVectorizer()

X_bow = vectorizer.fit_transform(data['clean_reviews'])
y = data['target_encoded']

nb_model = MultinomialNB()

cv_results = cross_validate(nb_model, X_bow, y, cv=5, scoring=['accuracy'])

mean_score = cv_results['test_accuracy'].mean()
print(f"5-Fold Çapraz Doğrulama Skorları: {cv_results['test_accuracy']}")
print(f"Ortalama Doğruluk Skoru: %{mean_score * 100:.2f}")

5-Fold Çapraz Doğrulama Skorları: [0.805  0.82   0.805  0.835  0.8175]
Ortalama Doğruluk Skoru: %81.65


## 3. N-gram Modellemesi

👀 Stop kelimeleri kaldırmamanızı istediğimizi hatırlayın. Neden? 

👉 Naive Bayes modelini bigramlarla eğiteceğiz. Bu nedenle, “I do not like coriander” (Kişnişi sevmiyorum) gibi bir cümlede, örneğin bu cümlede olumsuzluğu tespit etmek için “do not” bigramını taramak önemlidir.

❓ **Soru (bigramlarla NaiveBayes)** ❓

`cross_validate` kullanarak, metinlerin 2-gram Bag-of-Words temsilinde eğitilmiş bir Multinomial Naive Bayes modelini puanlayın.

In [12]:
vectorizer = CountVectorizer(ngram_range = (2,2))
naivebayes = MultinomialNB()

X_bow = vectorizer.fit_transform(data.clean_reviews)

cv_nb = cross_validate(
    naivebayes,
    X_bow,
    data.target_encoded,
    scoring = "accuracy"
)

round(cv_nb['test_score'].mean(),2)

0.84

🏁 Tebrikler! Artık vektörleştirilmiş metinler üzerinde Naive Bayes modelini nasıl eğiteceğinizi biliyorsunuz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

